# An-Ra V4 SFT — protected T4 child-lineage trainer

This notebook trains instruction-following behavior from an audited SFT dataset. It never modifies the foundation checkpoint. The default is a 15-minute pilot. Full mode requires a separate signed approval bound to the reviewed pilot checkpoint.

In [ ]:
# Operator configuration
WORKER_ROLE = 'canonical_trainer'
WORKER_ID = 'colab-t4-sft-primary'
RUN_MODE = 'pilot'  # Full mode also requires sft-v4/full_sft_approval.json.
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO_REF = 'iterate500'
# Pin the source that contains the complete SFT readiness gate. This is
# deliberately the previous pushed commit because this notebook change is
# itself part of the next commit (avoiding a self-referential pin).
REPO_COMMIT = 'a426d62bdfd77081d5358d2960442a472a18aed3'
SESSION_BUDGET_MINUTES = 15 if RUN_MODE == 'pilot' else 150
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
DRIVE_MOUNT_ROOT = '/content/drive'
assert RUN_MODE in {'pilot', 'full'}
print({'role': WORKER_ROLE, 'worker': WORKER_ID, 'mode': RUN_MODE, 'minutes': SESSION_BUDGET_MINUTES})

In [ ]:
# Mount the authorized Drive and require a real T4.
from google.colab import drive
drive.mount(DRIVE_MOUNT_ROOT, force_remount=True)
import os, pathlib, shutil, subprocess, json, hashlib, torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
assert 'T4' in gpu_name.upper(), f'Expected T4, received {gpu_name}'
subprocess.run(['nvidia-smi'], check=True)
print('READY:', gpu_name)

In [ ]:
# Clone the exact source branch.
REPO = pathlib.Path('/content/anra')
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', '-c', 'http.version=HTTP/1.1', 'clone', '--depth', '1', '--single-branch', '--branch', REPO_REF, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
# The branch tip is normally the pinned commit and is already in the depth-1
# clone. Only deepen when the pinned commit is older than that tip; avoid
# --unshallow because some Colab Git builds reject it on complete clones.
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if head != REPO_COMMIT:
    fetch = subprocess.run(['git', 'fetch', '--depth', '100', 'origin', REPO_REF], text=True)
    if fetch.returncode:
        raise RuntimeError(f'Unable to fetch source branch {REPO_REF}; check the Colab network and GitHub access.')
subprocess.run(['git', 'checkout', '--detach', REPO_COMMIT], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert commit == REPO_COMMIT, f'Unexpected source commit: {commit}'
assert not subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip()
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Clean source:', commit)

In [ ]:
# Locate the single shared home. The SFT assets are kept in its sft-v4 child folder.
from training.colab_shared_assets import resolve_colab_training_assets
ASSETS = resolve_colab_training_assets(DRIVE_MOUNT_ROOT, require_pack_parts=False, discovery_timeout_seconds=45)
TRAINING_HOME = pathlib.Path(ASSETS.training_home)
SFT_HOME = TRAINING_HOME / 'sft-v4'
BASE_CHECKPOINT = TRAINING_HOME / 'anra-v4-current-full-resume.pt'
DATASET_MANIFEST = SFT_HOME / 'sft-v4-train.manifest.json'
VALIDATION_MANIFEST = SFT_HOME / 'sft-v4-validation.manifest.json'
SOURCE_RECEIPT = SFT_HOME / 'sft-v4-source-receipts.json'
LINEAGE_MANIFEST = SFT_HOME / 'anra-v4-sft-lineage.json'
assert BASE_CHECKPOINT.is_file(), f'Missing foundation parent: {BASE_CHECKPOINT}'
assert DATASET_MANIFEST.is_file(), f'Missing audited SFT train manifest: {DATASET_MANIFEST}'
assert VALIDATION_MANIFEST.is_file(), f'Missing audited SFT validation manifest: {VALIDATION_MANIFEST}'
assert SOURCE_RECEIPT.is_file(), f'Missing hash-verified SFT source receipt: {SOURCE_RECEIPT}'
# Load the shared SFT key first so trusted Colab accounts can resume without re-entering it.
# A Colab Secret remains supported as a safer fallback when the shared key file is unavailable.
SFT_SIGNING_KEY_FILE = SFT_HOME / 'anra-sft-manifest-signing-key.json'
if SFT_SIGNING_KEY_FILE.is_file():
    # PowerShell-created JSON may contain a UTF-8 BOM; utf-8-sig accepts
    # both BOM and BOM-free files without changing the key bytes.
    key_payload = json.loads(SFT_SIGNING_KEY_FILE.read_text(encoding='utf-8-sig'))
    shared_key = str(key_payload.get('key', '')).strip()
    assert len(shared_key) >= 32, f'Invalid shared SFT signing key: {SFT_SIGNING_KEY_FILE}'
    os.environ['ANRA_MANIFEST_SIGNING_KEY'] = shared_key
elif not os.environ.get('ANRA_MANIFEST_SIGNING_KEY'):
    from getpass import getpass
    os.environ['ANRA_MANIFEST_SIGNING_KEY'] = getpass('Campaign manifest key: ').strip()
assert len(os.environ['ANRA_MANIFEST_SIGNING_KEY']) >= 32, 'A valid SFT signing key is required.'
SCRATCH = pathlib.Path('/content/anra-sft-scratch')
SCRATCH.mkdir(exist_ok=True)
print({'training_home': str(TRAINING_HOME), 'sft_home': str(SFT_HOME), 'parent_step': ASSETS.vault_step})

In [ ]:
# Create the signed SFT lineage once, then preflight and run. A changed
# validation split is a new immutable lineage: archive prior pilot evidence
# instead of trying to resume it under different hashes.
LINEAGE_ID = 'anra-v4-sft-003'
def _sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def _archive_current_lineage(previous_id):
    archive_root = SFT_HOME / 'archive'
    archive = archive_root / f'{previous_id}-{REPO_COMMIT[:8]}'
    suffix = 2
    while archive.exists():
        archive = archive_root / f'{previous_id}-{REPO_COMMIT[:8]}-{suffix}'
        suffix += 1
    archive.mkdir(parents=True, exist_ok=True)
    for name in ('anra-v4-sft-lineage.json', 'anra-v4-current-full-resume.pt',
                  'anra-v4-current-full-resume.json', 'latest_sft_report.json',
                  'ready_to_sft.json', 'full_sft_approval.json'):
        source = SFT_HOME / name
        if source.exists():
            shutil.move(str(source), str(archive / name))
    print({'archived_lineage': previous_id, 'archive': str(archive)})

# A signed lineage is immutable. Roll it forward whenever the source commit,
# dataset, validation split, or foundation parent changed. This prevents a
# stale Drive manifest from producing the opaque preflight exit-1 seen in
# Colab after a notebook or data update.
if LINEAGE_MANIFEST.exists():
    previous = json.loads(LINEAGE_MANIFEST.read_text(encoding='utf-8-sig'))
    previous_id = str(previous.get('lineage_id', 'legacy'))
    previous_dataset = previous.get('dataset') if isinstance(previous.get('dataset'), dict) else {}
    previous_evaluation = previous.get('evaluation') if isinstance(previous.get('evaluation'), dict) else {}
    mismatch = (
        previous_id != LINEAGE_ID
        or str(previous.get('source_commit', '')) != REPO_COMMIT
        or str(previous_dataset.get('manifest_sha256', '')) != _sha256(DATASET_MANIFEST)
        or str(previous_evaluation.get('manifest_sha256', '')) != _sha256(VALIDATION_MANIFEST)
        or str((previous.get('parent') or {}).get('base_checkpoint_sha256', '')) != _sha256(BASE_CHECKPOINT)
    )
    if mismatch:
        _archive_current_lineage(previous_id)
if not LINEAGE_MANIFEST.exists():
    subprocess.run([
        'python', '-m', 'training.sft_v4', 'prepare-lineage',
        '--lineage-id', LINEAGE_ID,
        '--dataset-manifest', str(DATASET_MANIFEST),
        '--validation-manifest', str(VALIDATION_MANIFEST),
        '--source-receipt', str(SOURCE_RECEIPT),
        '--base-checkpoint', str(BASE_CHECKPOINT),
        '--output', str(LINEAGE_MANIFEST),
    ], check=True)
common = [
    'python', '-u', '-m', 'training.sft_v4',
    '--dataset-manifest', str(DATASET_MANIFEST),
    '--validation-manifest', str(VALIDATION_MANIFEST),
    '--source-receipt', str(SOURCE_RECEIPT),
    '--lineage-manifest', str(LINEAGE_MANIFEST),
    '--base-checkpoint', str(BASE_CHECKPOINT),
    '--vault-root', str(TRAINING_HOME),
    '--local-checkpoint', str(SCRATCH / 'anra-v4-sft-current-full-resume.pt'),
    '--mode', RUN_MODE, '--max-minutes', str(SESSION_BUDGET_MINUTES),
    '--batch-size', str(BATCH_SIZE), '--accumulation', str(GRADIENT_ACCUMULATION),
]
def _run_checked(label, command):
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end='')
    if result.returncode:
        if result.stderr:
            print(f'[{label} stderr]\n{result.stderr}', end='')
        raise RuntimeError(f'{label} failed with exit status {result.returncode}; see diagnostics above.')

_run_checked('SFT preflight', [*common[:4], 'preflight', *common[4:]])
if WORKER_ROLE == 'verify_only':
    print('Preflight complete; no weights modified.')
else:
    assert WORKER_ROLE == 'canonical_trainer'
    _run_checked('SFT run', [*common[:4], 'run', *common[4:]])
    print('SFT session complete. Review sft-v4/latest_sft_report.json; use approve-full before enabling RUN_MODE=full.')